# Retrieval & generation  with llama Stack & Milvus on wx.data


#### Prerequesites

- run one foundational model on RHOAI
- run one embedding model on RHOAI
- run Llama Stack Server server in version >=0.7.1 on RHOAI (using remote::vllm providers) with enabled inline Milvus
- documents embeded and added to a the Milvus collection - (see Indexing notebook wwith llama stack)

Add environment variables
- **LLAMASTACK_URL**
- **LLAMASTACK_APIKEY** if auth/authz is enabled on Llama Stack Server

NOTE: You can run also LLama Stack locally, using either remote::vllm (for remote models) or ollama (for local models) providers.


In [1]:
!pip install dotenv
!pip install llama_stack_client>=0.7.1

Looking in indexes: https://pypi.org/simple/, https://pypi.org/simple/


#### Import dependencies

**NOTE:** Loads LLAMASTACK_URL from `.env` file unless environment variables are pre-configured.

**NOTE:** Logging from httpx library set to WARNING level.

In [2]:
import os

# Load from .env only if LLAMASTACK_URL not already set
if not os.getenv('LLAMASTACK_URL'):
    from dotenv import load_dotenv
    load_dotenv()

import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

from llama_stack_client import LlamaStackClient

#### Create LlamaStackClient object

**NOTE:** If you do not have authorization enabled on your Llama Stack instance, just skip providing `LLAMASTACK_APIKEY`.


In [3]:
client = LlamaStackClient(
    base_url=os.getenv("LLAMASTACK_URL", "http://localhost:5321"),
    api_key=os.getenv("LLAMASTACK_APIKEY")
)


#### Models used in the example

In [4]:
# Read model names from environment variables with fallbacks
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "vllm-embedding/nomic-ai/nomic-embed-text-v1.5")
EMBEDDING_MODEL_DIMENSION = int(os.getenv("EMBEDDING_MODEL_DIMENSION", "768"))

LLM_MODEL = os.getenv("LLM_MODEL", "vllm-inference/llama-3-2-3b")

# Vector store ID can be set via environment or detected from indexing script
VECTOR_STORE_ID = os.getenv("VECTOR_STORE_ID", None)

In [5]:
client.models.list()[0].model_dump()

{'id': 'vllm-embedding/nomic-ai/nomic-embed-text-v1.5',
 'created': 1776949343,
 'owned_by': 'llama_stack',
 'custom_metadata': {'model_type': 'embedding',
  'provider_id': 'vllm-embedding',
  'provider_resource_id': 'nomic-ai/nomic-embed-text-v1.5',
  'embedding_dimension': 768},
 'object': 'model'}

#### Tests model availability via llama stack

In [6]:
msgs = [{"role": "user", "content": "Generate a short poem"}]

llm_resp = client.chat.completions.create(
    messages=msgs,
    model=LLM_MODEL
)

llm_resp.model_dump()

{'id': 'chatcmpl-e956a1892182446fba5f7c8c1ec87294',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'message': {'annotations': None,
    'audio': None,
    'content': 'Here\'s a short poem:\n\n"Moonlit whispers in the night\nA gentle breeze, a soft delight\nStars above, a twinkling sea\nA peaceful world, for you and me"',
    'function_call': None,
    'refusal': None,
    'role': 'assistant',
    'tool_calls': None,
    'reasoning_content': None},
   'logprobs': None,
   'stop_reason': None}],
 'created': 1776949344,
 'model': 'vllm-inference/llama-3-2-3b',
 'object': 'chat.completion',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': {'completion_tokens': 40,
  'completion_tokens_details': None,
  'prompt_tokens': 39,
  'prompt_tokens_details': None,
  'total_tokens': 79},
 'prompt_logprobs': None,
 'kv_transfer_params': None}

In [7]:
emb_response = client.embeddings.create(input="Hello", model=f"{EMBEDDING_MODEL}")
print(f"Embedding dimension: {len(emb_response.data[0].embedding)}")
print(f"Model: {emb_response.model}")

Embedding dimension: 768
Model: vllm-embedding/nomic-ai/nomic-embed-text-v1.5


### Files used as benchmarking data

**NOTE:** The below example assumes that benchmarking data resides in subfolder: `data/`. Moreover, this example itself does not deliver these artifacts - you need to replace:
- `benchmark_qa_path` with filename of benchmarking dataset, against which we'll test retrieval

In [8]:
from pathlib import Path
import json

In [9]:
docs_folder_path = Path("data/")
benchmark_qa_path = docs_folder_path.joinpath("ibm_annual_report_pdf_benchmarking_data.json")

In [10]:
benchmark_qa = json.loads(benchmark_qa_path.read_bytes())
benchmark_qa[:3]

[{'correct_answer_document_ids': ['ibm-annual-report-2024-pt1_1-20.pdf'],
  'question': 'What was the operating other income and expense in 2024?',
  'correct_answer': '$1,656 million'},
 {'correct_answer_document_ids': ['ibm-annual-report-2024-pt1_1-20.pdf'],
  'question': 'What is the amount of gain on land/building dispositions included in "Other"',
  'correct_answer': '$126 million'},
 {'correct_answer_document_ids': ['ibm-annual-report-2024-pt1_1-20.pdf'],
  'question': 'How much was the non-operating retirement-related costs in the current-year period?',
  'correct_answer': '$3,457 million'}]

### Choosing vector store with knowledge base

**NOTE:** Use `llamastack_rag_index_building.ipynb` notebook to build knowledge base, stored in vector store - reuse the same vector store ID in this example.

In [11]:
vector_stores = client.vector_stores.list()
print(f"Found {len(vector_stores.data)} vector store(s)")
if vector_stores.data:
    print(f"Most recent ID: {vector_stores.data[0].id}")

Found 5 vector store(s)
Most recent ID: vs_cb0bb787-ddec-40bb-8bab-b51ccf08fec5


In [12]:
# Use vector store ID from environment if set, otherwise use the first available
if VECTOR_STORE_ID:
    vector_store_id = VECTOR_STORE_ID
else:
    # List available vector stores and use the most recent one
    vector_stores = client.vector_stores.list()
    if vector_stores.data:
        vector_store_id = vector_stores.data[0].id
        print(f"Using vector store: {vector_store_id}")
    else:
        raise ValueError("No vector stores found. Please run the indexing notebook first.")

vector_store_id

Using vector store: vs_cb0bb787-ddec-40bb-8bab-b51ccf08fec5


'vs_cb0bb787-ddec-40bb-8bab-b51ccf08fec5'

### Build user prompt

In [13]:
DEFAULT_PROMPT_TEMPLATE = "{reference_documents}\n[conversation]: {question}. Answer with no more than 150 words. If you cannot base your answer on the given document, please state that you do not have an answer. Respond exclusively in the language of the question, regardless of any other language used in the provided context. Ensure that your entire response is in the same language as the question.\n"

DEFAULT_CONTEXT_TEMPLATE = "[document]: {document}\n"

DEFAULT_SYSTEM_PROMPT = "You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature.\nIf a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you don’t know the answer to a question, please don’t share false information.\n"

In [14]:
def build_prompt(
        question: str,
        reference_documents: list[str]=None,
        prompt_template_text: str = DEFAULT_PROMPT_TEMPLATE,
        context_template_text: str = DEFAULT_CONTEXT_TEMPLATE,
) -> str:
    """
    Warning: It's simplified prompt builder, without sampling of the reference documents
    """
    if reference_documents:
        reference_documents = [
            context_template_text.format(document=reference_document)
            for reference_document in reference_documents
        ]
    else:
        reference_documents = []
    prompt_variables = {
        "question": question,
        "reference_documents": "\n".join(reference_documents)
    }
    return prompt_template_text.format(**prompt_variables)

### Retrieval & generation

In [15]:
example_question = benchmark_qa[-1].get("question")
example_question

'What was the total revenue for year 2024?'

In [16]:
search_response = client.vector_stores.search(
    vector_store_id=vector_store_id,
    query=example_question,
    max_num_results=5,
)
search_response.model_dump()


{'data': [{'content': [{'text': 'Financial Performance Summary                        \nIn 2024, we reported $62.8 billion in revenue, income from continuing operations of $6.0 billion, which includes the impact of the',
     'type': 'text',
     'chunk_metadata': None,
     'embedding': None,
     'metadata': None}],
   'file_id': 'ibm-annual-report-2024-pt1_1-20.pdf',
   'filename': '',
   'score': 0.8112797141075134,
   'attributes': {'document_id': 'ibm-annual-report-2024-pt1_1-20.pdf',
    'start_index': 29913.0,
    'sequence_number': 166.0}},
  {'content': [{'text': 'Transaction Processing revenue of $8,277 million increased 8.7 percent as reported (9.6 percent adjusted for currency) in 2024',
     'type': 'text',
     'chunk_metadata': None,
     'embedding': None,
     'metadata': None}],
   'file_id': 'ibm-annual-report-2024-pt1_1-20.pdf',
   'filename': '',
   'score': 0.7954844236373901,
   'attributes': {'document_id': 'ibm-annual-report-2024-pt1_1-20.pdf',
    'start_inde

In [17]:
reference_documents = [doc.content[0].text for doc in search_response.data]
len(reference_documents)

5

In [18]:
user_prompt = build_prompt(question=example_question, reference_documents=reference_documents)

In [19]:
msgs = [
    {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt}
]


response_chat = client.chat.completions.create(
    model=LLM_MODEL,
    messages=msgs
)
answer = response_chat.choices[0].message.content
answer

'According to the document, the total revenue for 2024 was $62.8 billion.'